## Streszczenie dla kierownictwa

Zespół operacji logistycznych planuje randomizowane doświadczenie polowe porównujące trzy strategie trasowania kierowców (statyczna linia bazowa, dynamiczne przetrasowanie, zoptymalizowana przez AI) w trzech regionach depot, z uśrednionym opóźnieniem dostawy (w minutach) jako odpowiedzią. PROC GLMPOWER pobiera *przykładowy* zbiór danych z przypuszczonych średnich komórkowych i rozwiązuje, ile łącznie kierowców potrzeba, aby wykryć efekty strategii przy mocy 80% i 90%, a następnie pokazuje, jak osiągalna moc maleje wraz ze wzrostem zmienności między trasami.

# Dobór wielkości próby dla doświadczenia polowego trasowania kierowców za pomocą PROC GLMPOWER

## Streszczenie dla kierownictwa

Zespół operacji logistycznych szykuje się do uruchomienia randomizowanego doświadczenia polowego porównującego trzy strategie trasowania kierowców -- linię bazową **Statyczną**, system dynamicznego przetrasowania **Dynamiczny**, oraz planer **Zoptymalizowany przez AI** -- wdrożone w trzech regionach depot (Northeast, Midwest, West). Odpowiedzią jest uśrednione **opóźnienie dostawy w minutach**. Przed zaangażowaniem pojemności floty w doświadczenie zespół musi odpowiedzieć: *ilu kierowców potrzebujemy, aby wiarygodnie wykryć oczekiwaną poprawę operacyjną?*

Ten notatnik wykorzystuje **PROC GLMPOWER** do przeprowadzenia prospektywnej analizy mocy i wielkości próby dla ogólnego modelu liniowego leżącego u podstaw doświadczenia. Wychodząc od *przykładowego* zbioru danych przypuszczonych średnich komórkowych, (1) rozwiązuje całkowitą liczebność potrzebną do osiągnięcia mocy 80% i 90% dla ogólnych efektów strategii i regionu, (2) pokazuje, jak osiągalna moc maleje wraz ze wzrostem zmienności między trasami, oraz (3) tworzy krzywą mocy wspierającą decyzję o liczebności próby.

> **Kluczowy wniosek:** Przy przypuszczalnym odchyleniu standardowym błędu 8 minut, około **63 kierowców** zapewnia moc 80%, a **83 kierowców** zapewnia moc 90% dla wykrycia efektów strategii trasowania -- ale jeśli zmienność opóźnienia zbliża się do 10 minut, nawet 90 zaangażowanych kierowców nie osiąga mocy 70%, co podkreśla wartość ścisłych, dobrze zinstrumentowanych tras.

---
## 1. Budowa przykładowego planu

Pierwszy krok koduje układ doświadczenia i *przypuszczone* przez zespół średnie opóźnienie dla każdej kombinacji strategia trasowania x region depot. Ustalamy ziarno losowe i dodajemy znikomy szum, aby średnie wyglądały realistycznie, zachowując zrównoważoną strukturę 3x3. Waga `cell_n` równa 1 w każdej komórce informuje GLMPOWER, że plan jest zrównoważony.

In [1]:
/* ================================================================
   Generowanie przykładowego zbioru danych przypuszczonych średnich opóźnień.
   Jeden wiersz na komórkę planu strategia-trasowania x region-depot.
   ================================================================ */
DANE routing_trial;
   CALL streaminit(20260531);
   DŁUGOŚĆ routing_strategy $8 depot_region $9;
   TABLICA strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   TABLICA region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   TABLICA smean[3]     _temporary_ (18.0 14.5 11.0);   /* przypuszczone średnie strategii */
   TABLICA radj[3]      _temporary_ (1.5  0.0 -1.0);    /* korekty regionalne (min)  */
   POWTÓRZ i = 1 TO 3;
      POWTÓRZ j = 1 TO 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         WYJŚCIE;
      KONIEC;
   KONIEC;
   USUŃ i j;
WYKONAJ;

PROCEDURA DRUKUJ DANE=routing_trial ETYKIETA noobs;
   ZMIENNA routing_strategy depot_region mean_delay_min cell_n;
   ETYKIETA routing_strategy="Strategia trasowania" depot_region="Region depot"
         mean_delay_min="Uśrednione opóźnienie dostawy (min)" cell_n="Waga komórki";
   TYTUŁ "Przykładowe średnie komórkowe: przypuszczone opóźnienie dostawy (minuty)";
WYKONAJ;

                        Przykładowe średnie komórkowe: przypuszczone opóźnienie dostawy (minuty)                        

Strategia trasowania  Region depot     Uśrednione opóźnienie dostawy (min)   Waga komórki
Static                Northeast                               19.687507651              1
Static                Midwest                                17.9871067398              1
Static                West                                   16.8240210086              1
Dynamic               Northeast                              15.9537535365              1
Dynamic               Midwest                                14.4415169858              1
Dynamic               West                                   12.8636389493              1
AIOpt                 Northeast                              12.6143811724              1
AIOpt                 Midwest                                 10.893885771              1
AIOpt                 West                                     9.635


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Wielkość próby dla planu ogólnego

Mając plan, prosimy GLMPOWER o **rozwiązanie całkowitej wielkości próby** (`NTOTAL = .`) przy mocy 80% i 90%. Instrukcja `MODEL` określa układ dwuczynnikowy z interakcją (`routing_strategy*depot_region`), `WEIGHT cell_n` deklaruje zrównoważone wagi profilu, a `STDDEV = 8` to zakładany pierwiastek błędu średniokwadratowego opóźnienia dostawy. `NFRACTIONAL` pozwala procedurze raportować dokładne liczby ułamkowe przed zaokrągleniem w górę.

Rejestrujemy z góry również trzy zaplanowane porównania `CONTRAST` -- AI-Opt vs. Statyczna, Dynamiczna vs. Statyczna oraz dowolne przetrasowanie vs. Statyczna -- które dokumentują operacyjnie znaczące hipotezy, do przetestowania których zbudowano doświadczenie.

In [2]:
/* ================================================================
   Rozwiązanie liczby kierowców potrzebnej do osiągnięcia mocy 80% i 90%
   dla efektów strategii trasowania i regionu depot.
   ================================================================ */
PROCEDURA glmpower DANE=routing_trial;
   KLASA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   WAGA cell_n;
   ETYKIETA routing_strategy="Strategia trasowania" depot_region="Region depot"
         mean_delay_min="Uśrednione opóźnienie dostawy (min)";
   CONTRAST "AI-Opt vs. Statyczna"                routing_strategy -1  0  1;
   CONTRAST "Dynamiczna vs. Statyczna"             routing_strategy -1  1  0;
   CONTRAST "Dowolne przetrasowanie vs. Statyczna" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TYTUŁ "Wielkość próby do wykrycia efektów strategii trasowania na opóźnienie";
WYKONAJ;

                        Przykładowe średnie komórkowe: przypuszczone opóźnienie dostawy (minuty)                        

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Uśrednione opóźnienie dostawy (min)   
Source              Strategia trasowania                  
Source              Region depot                          
Source              Strategia trasowania*Region depot     

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Czułość mocy na zmienność i wielkość doświadczenia

Odpowiedź na pytanie o wielkość próby zależy od zakładanego odchylenia standardowego błędu, które rzadko jest znane z góry precyzyjnie. Tutaj odwracamy pytanie: **ustalamy** kilka kandydujących sumarycznych liczebności (`NTOTAL = 45 90 180`) i **rozwiązujemy dla osiągniętej mocy** (`POWER = .`) na siatce przypuszczalnych odchyleń standardowych opóźnienia (6, 8 i 10 minut). Wynikowa tabela jest mapą czułości -- pokazuje, jak odporny jest każdy plan liczebności, jeśli rzeczywista zmienność tras okaże się wyższa niż oczekiwano.

In [3]:
/* ================================================================
   Siatka czułości: osiągnięta moc na kandydujących wielkościach
   doświadczenia i przypuszczalnych odchyleniach standardowych opóźnienia.
   ================================================================ */
PROCEDURA glmpower DANE=routing_trial;
   KLASA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   WAGA cell_n;
   ETYKIETA routing_strategy="Strategia trasowania" depot_region="Region depot"
         mean_delay_min="Uśrednione opóźnienie dostawy (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TYTUŁ "Osiągnięta moc wg scenariuszy zmienności i wielkości doświadczenia";
WYKONAJ;

                        Przykładowe średnie komórkowe: przypuszczone opóźnienie dostawy (minuty)                        

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Uśrednione opóźnienie dostawy (min)   
Source              Strategia trasowania                  
Source              Region depot                          

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Krzywa mocy dla decyzji o liczebności próby

Na koniec wykreślamy **krzywą mocy** -- osiągniętą moc w miarę wzrostu całkowitej liczebności od 30 do 270 kierowców w krokach co 30 -- dla modelu strategia-plus-region przy oczekiwanym odchyleniu standardowym 8 minut. Rozwiązanie `POWER = .` na tej siatce liczebności generuje krzywą jako stabelaryzowaną serię `N Total` vs. `Power`, dzięki czemu możemy odczytać liczebność, przy której przekraczany jest każdy z konwencjonalnych progów 80% i 90%.

In [4]:
/* ================================================================
   Krzywa mocy: osiągnięta moc vs. całkowita liczba zaangażowanych
   kierowców, przemiatana od 30 do 270 w krokach co 30 przy
   oczekiwanym odchyleniu standardowym 8 min.
   ================================================================ */
PROCEDURA glmpower DANE=routing_trial;
   KLASA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   WAGA cell_n;
   ETYKIETA routing_strategy="Strategia trasowania" depot_region="Region depot"
         mean_delay_min="Uśrednione opóźnienie dostawy (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TYTUŁ "Krzywa mocy: osiągnięta moc vs. całkowita liczba zaangażowanych kierowców";
WYKONAJ;

                        Przykładowe średnie komórkowe: przypuszczone opóźnienie dostawy (minuty)                        

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Uśrednione opóźnienie dostawy (min)   
Source              Strategia trasowania                  
Source              Region depot                          

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretacja i rekomendacja

Analiza daje zespołowi operacyjnemu obronny plan liczebności próby:

- **Dobór bazowy (Sekcja 2).** Zakładając 8-minutowe resztkowe odchylenie standardowe, pełny model dwuczynnikowy (strategia, region i ich interakcja) osiąga **moc 80% przy 63 kierowcach** i **moc 90% przy 83 kierowcach**. Zaokrąglając w górę na wypadek odpadnięć, cel liczebności bliski **90 kierowców** komfortowo przekracza próg 90%.

- **Czułość ma znaczenie (Sekcja 3).** Moc jest wysoce czuła na zmienność opóźnienia. Przy 90 kierowcach osiągnięta moc spada z ~99% (SD=6) przez ~87% (SD=8) do ~68% (SD=10). Pilotaż na 45 kierowcach jest wystarczający tylko wtedy, gdy zmienność jest niska (~81% mocy przy SD=6), a jest wyraźnie niedomocowany (~37%), gdy SD sięga 10. Praktyczna implikacja: inwestowanie w spójne, dobrze zinstrumentowane trasy w celu ograniczenia zmienności jest tak samo cenne jak dodawanie kierowców.

- **Krzywa mocy (Sekcja 4).** Wykreślona dla modelu strategia-plus-region (bez członu interakcji, optyki użytej do przemiatania czułości), osiągnięta moc wzrasta z 0,37 przy 30 kierowcach do 0,69 przy 60, 0,87 przy 90 i 0,95 przy 120, spłaszczając się powyżej 0,99 przy 180. Odczytując krzywą względem celów, moc 80% pojawia się przy około 77 kierowcach, a 90% przy około 99 -- nieco wyżej niż dobór pełnego modelu w Sekcji 2, ponieważ usunięcie członu interakcji rozkłada ten sam efekt na mniejszą liczbę stopni swobody modelu.

**Rekomendacja:** Zaangażuj około **90 kierowców** (30 na strategię trasowania, zrównoważonych w trzech regionach depot). To przekracza moc 90% dla pełnego modelu przy oczekiwanym 8-minutowym odchyleniu standardowym, utrzymuje ~87% mocy nawet na bardziej konserwatywnej krzywej modelu zredukowanego, i utrzymuje doświadczenie na tyle małe, aby wykonać je w ciągu jednego kwartału operacyjnego.

*Uwaga:* GLMPOWER analizuje przypuszczony *plan*, a nie zaobserwowane wyniki -- więc wiarygodność tych liczb opiera się na przypuszczonych średnich i odchyleniu standardowym. Zespoły powinny ponownie ocenić dobór liczebności, gdy krótki pilotaż dostarczy empirycznego oszacowania zmienności opóźnienia dostawy.